In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

#pd.set_option('display.max_rows', 1000); pd.set_option('display.max_columns', 1000); pd.set_option('display.width', 1000)


# Load data

In [ ]:
df = pd.read_csv("/Users/AdenHsu/Documents/DataFolder/MACSS-FULLDemogData_2026-01-14_deidentified_SB(in).csv")

# Description

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.describe

In [ ]:
df.info

# Data Preparation

In [ ]:
# strip unnecessary spaces off the columns
df.columns = df.columns.str.strip()

# rename the columns
df = df.rename(columns = {
    "record_id":"id",
    "Religion":"religion",
    "Ethnicity":"hispanic", 
    "What is your annual household income?":"income",
    'Black or African American':'black',
    'Asian':'asian',
    'White':'white',
    'Native Hawaiian or Other Pacific Islander':'pacific islander',
    'Native America/Alaska Native':'native american',
    'Other':'other',
    'Current medications:':'medication',
    'ASD (formal diagnosis) (choice=Participant)':"asd",
    'ASD traits (choice=Participant)':'asd_traits',
    'Developmental Delay (choice=Participant)':"developmental_delay",
    'Intellectual Disability (choice=Participant)':'intellectual_disability',
    'ADHD (choice=Participant)':'adhd',
    'Learning Disability (choice=Participant)':'learning_disability',
    'Depression (choice=Participant)':'depression',
    'Anxiety (choice=Participant)':'anxiety',
    "OCD/Tics/Tourette's (choice=Participant)":'ocd_tics_tourettes',
    'Bipolar Disorder (choice=Participant)':'bipolar',
    'Schizophrenia/psychosis (choice=Participant)' : 'schizophrenia',
    'Eating Disorder (choice=Participant)':'eating_disorder',
    'Seizures (choice=Participant)':'seizures',
    'Genetic Syndromes (choice=Participant)':'genetic_syndromes',
    'Other: (choice=Participant)':'other_condition',
    'other':'other_race',
    })

# create new age column to replace age_months and age_year
if 'age_months' in df.columns:
    df['age'] = df['age_months'] / 12
    df['age'] = df['age'].round(2)

# drop unnecessary columns
if 'age_years' in df.columns:
    df = df.drop(['age_years',
        'age_months',
        'gender: Other (Please Specify)',
        'Other (please specify):',
        'Other (please specify):.1',
        'srs_sex',
        'srs_sex_other',
        'srs_date',
        'adost_exam_date',
        'ados_examiner',
        'ados_exact_age',
        'ados_exact_age_months',
        'body_perception_questionnaire_complete',
        'srs_age_months',
        'wasi_date',
        'wasi_age_months',
        ],axis=1,errors='ignore')
df.insert(1, "age", df.pop("age"))
df['gender'] = df['gender'].replace({'Transgender Man':'Male',
                                     'Gender Variant/Non-Binary':'Other'})
df['hispanic'] = df['hispanic'].replace({'Not Hispanic/Latino':0,
                                         'Hispanic/Latino':1})
df['dx'] = df['dx'].replace({'TD':0,'ASD':1})

# change income to a numerical variable
df['income'] = df['income'].replace({
    "$0 - $15,000":7500,
    '$15,000 - $24,999':20000,
    '$25,000 - $34,999':30000,
    "$35,000 - $49,999":42500,
    "$50,000 - $74,999":62500,
    '$75,000 - $99,999':82500,
    "$100,000 - $149,999":125000,
    "$150,000 - $199,999":175000,
    "$200,000 and over":200000}).astype(float)

# drop medication column
if 'medication' in df.columns:
    df = df.drop('medication',axis=1)
# change religion column to religion y/n
df['religion'] = df['religion'].fillna(0)
df['religion'] = df['religion'].replace({"none":0,
                                         "NA ":0,
})
df['religion'] = (df['religion'] != 0).astype(int)

# there's one row at the end that is just n/a for everything - drop that
df = df.drop(index=61)

In [ ]:
display(df.isnull().sum())
# majority of the null rows come from wasi and ados. ados is essentiall used to diagnose autism, so don't use that for prediction. wasi will be imputed with knn?

In [ ]:
#df['dx'].value_counts()
print(f"There are {df['dx'].value_counts()[0]} instances of TD and {df['dx'].value_counts()[1]} instances of ASD")
print(f"TD makes up {df['dx'].value_counts()[0]/(19+42) * 100}% of the dataset and ASD makes up {df['dx'].value_counts()[1]/(19+42) * 100}%")

In [ ]:
demographic_columns = (['age','sex','religion','hispanic','income','black','asian','white','pacific islander','native american','other'])
clinical_columns = (['developmental_delay',
                     'intellectual_disability',
                     'adhd','learning_disability',
                     'depression','anxiety',
                     'ocd_tics_tourettes',
                     'bipolar','schizophrenia',
                     'eating_disorder',
                     'seizures',
                     'genetic_syndromes',
                     'other_condition'])
ados_columns = ([])
bpq_columns = ([])
srs_columns = []
wasi_columns = []

# organize each column into sections
for i in df.columns:
    if i[0:3] == 'ado':
        ados_columns.append(i)
    elif i[0:3] == 'bpq':
        bpq_columns.append(i)
    elif i[0:3] == 'srs':
        srs_columns.append(i)
    elif i[0:3] == 'was':
        wasi_columns.append(i)
training_columns = (demographic_columns + clinical_columns+bpq_columns+wasi_columns+['ans_connect_postq'])

In [ ]:
y = df['dx']
X = df[training_columns]

# Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify = y)

In [ ]:
# convert object to pandas category
for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = X_train[col].astype('category')

for col in X_test.select_dtypes(include='object').columns:
    X_test[col] = X_test[col].astype('category')

# use knn imputer after split to avoid leakage
imputer = KNNImputer(n_neighbors=3)

# use knn imputer to impute one missing income variable
X_train[['income']] = imputer.fit_transform(X_train[['income']])
X_test[['income']]  = imputer.transform(X_test[['income']])

# replace all missing wasi with knn
X_train[wasi_columns] = imputer.fit_transform(X_train[wasi_columns])
X_test[wasi_columns]  = imputer.transform(X_test[wasi_columns])

# one-hot
cat_cols = X_train.select_dtypes(include=['category','object']).columns

X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)

# align
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# scale for use in logreg and other distance based models
scaler = StandardScaler()
X_train_enc_scaled = scaler.fit_transform(X_train_enc)
X_test_enc_scaled = scaler.transform(X_test_enc)

In [ ]:
# XGB classifier
model = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    enable_categorical=True
)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_enc, y_train)
y_pred = rf_model.predict(X_test_enc)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# feature importance
importances = rf_model.feature_importances_
feature_names = X_train.columns

feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).head(20)  # show top 20

# plot
plt.figure(figsize=(10, 6))
plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1])  # reversed for descending order
plt.xlabel("Feature Importance")
plt.title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
# define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

roc_data = {}
lift_data = {}
percentile_grid = np.linspace(0, 1, len(y_test))


# establish parameter grids
param_grids = {
    "Logistic Regression": {
        'clf__C': [0.01, 0.1, 1, 10]
    },
    "Random Forest": {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4]
    },
    "XGBoost": {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 5, 7],
        'clf__learning_rate': [0.01, 0.1, 0.2]
    },
    "SVM": {
        'clf__C': [0.1, 1, 10],
        'clf__kernel': ['linear', 'rbf']
    },
    "KNN": {
        'clf__n_neighbors': [3, 5, 7],
        'clf__weights': ['uniform', 'distance']
    }
}

pipelines = {}
for name, model in models.items():
    if name in ["Logistic Regression", "SVM", "KNN"]:
        pipelines[name] = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', model)
        ])
    else:
        pipelines[name] = Pipeline([
            ('clf', model)
        ])
# 5 fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in pipelines.items():
    print(f"\n=== {name} ===")

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring='balanced_accuracy',
        cv=cv,
        n_jobs=-1
    )

    grid.fit(X_train_enc, y_train)  # always use X_train_enc here

    # best model
    best_model = grid.best_estimator_

    # predict on test set
    y_pred = best_model.predict(X_test_enc)
    y_proba = best_model.predict_proba(X_test_enc)[:, 1]

    print("Best parameters:", grid.best_params_)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, y_proba))
    # Store ROC data
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    roc_data[name] = (fpr, tpr, roc_auc)

    # ----- Lift curve -----
    sorted_indices = np.argsort(y_proba)[::-1]
    y_true_sorted = y_test.values[sorted_indices]

    cumulative_true = np.cumsum(y_true_sorted)
    total_positives = np.sum(y_true_sorted)

    positive_rate = total_positives / len(y_true_sorted)
    lift = cumulative_true / (np.arange(1, len(y_true_sorted)+1) * positive_rate)

    lift_interp = np.interp(
        percentile_grid,
        np.arange(1, len(lift)+1)/len(lift),
        lift
    )

    lift_data[name] = lift_interp

In [ ]:
plt.figure(figsize=(8,6))

for name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Models")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

for name, lift_curve in lift_data.items():
    plt.plot(percentile_grid, lift_curve, label=name)

plt.plot([0,1], [1,1], 'k--')
plt.xlabel("Percent of Population")
plt.ylabel("Lift")
plt.title("Lift Curves for All Models")
plt.legend()
plt.grid(True)
plt.show()